# RPG — formar party por habilidade

Este notebook usa o mesmo mecanismo do `example.ipynb`: personagens têm classes e habilidades, as tarefas exigem requisitos e o sistema decide a melhor equipe.

In [1]:
from itertools import combinations

# Classes e habilidades
personagens = {
    'Aria': {'classe': 'Guerreira', 'habilidades': {'força', 'resistência'}},
    'Bruno': {'classe': 'Curandeiro', 'habilidades': {'cura', 'suporte'}},
    'Cleo': {'classe': 'Ladina', 'habilidades': {'agilidade', 'furtividade'}},
    'Dara': {'classe': 'Maga', 'habilidades': {'magia', 'inteligencia'}},
    'Eros': {'classe': 'Tank', 'habilidades': {'resistência', 'força'}},
}

# Tarefas do dungeon
missões = [
    ('M1', {'força'}),
    ('M2', {'cura'}),
    ('M3', {'agilidade'}),
]

# Compromisso de missão já acertado
compromisso_m1 = {'missao': 'M1', 'responsavel': 'Aria', 'estado': 'alocada'}

entrada = {
    'compromisso_m1': compromisso_m1,
    'ativos': ['Aria', 'Bruno', 'Cleo', 'Dara', 'Eros'],
    'disponiveis_para_final': ['Aria', 'Bruno', 'Cleo', 'Dara'],
}


def formar_coalizao(participantes, requisitos):
    for dupla in combinations(sorted(participantes), 2):
        p1, p2 = dupla
        cobertura = set(personagens[p1]['habilidades']) | set(personagens[p2]['habilidades'])
        if requisitos.issubset(cobertura):
            return [p1, p2]
    return []

print('Entradas:', entrada)
print('Dupla para missão final:', formar_coalizao(entrada['disponiveis_para_final'], {'força', 'cura'}))

Entradas: {'compromisso_m1': {'missao': 'M1', 'responsavel': 'Aria', 'estado': 'alocada'}, 'ativos': ['Aria', 'Bruno', 'Cleo', 'Dara', 'Eros'], 'disponiveis_para_final': ['Aria', 'Bruno', 'Cleo', 'Dara']}
Dupla para missão final: ['Aria', 'Bruno']


## 1. Alocar as missões

A função abaixo escolhe quem vai realizar cada missão, respeitando o compromisso já confirmado e distribuindo a carga entre os personagens.

In [2]:
def alocar(entrada):
    carga = {nome: 0 for nome in entrada['ativos']}
    plano = []

    preferida = None
    compromisso = entrada['compromisso_m1']
    if compromisso is not None and compromisso.get('estado') == 'alocada':
        preferida = compromisso['responsavel']

    for missao_id, requisito in missões:
        aptos = []
        for nome in sorted(carga):
            if requisito.issubset(personagens[nome]['habilidades']):
                aptos.append(nome)

        escolhida = None
        if missao_id == 'M1':
            if preferida in aptos:
                escolhida = preferida
        else:
            for nome in aptos:
                if escolhida is None or carga[nome] < carga[escolhida]:
                    escolhida = nome

        equipe = []
        estado = 'sem_capacidade'

        if missao_id == 'M1':
            estado = 'sem_acordo'

        if escolhida is not None:
            equipe.append(escolhida)
            carga[escolhida] += 1
            estado = 'alocada'

        plano.append({'missao': missao_id, 'equipe': equipe, 'estado': estado})

    resposta = entrada.copy()
    resposta['plano'] = plano
    return resposta

resultado = alocar(entrada)
for item in resultado['plano']:
    print(item)

{'missao': 'M1', 'equipe': ['Aria'], 'estado': 'alocada'}
{'missao': 'M2', 'equipe': ['Bruno'], 'estado': 'alocada'}
{'missao': 'M3', 'equipe': ['Cleo'], 'estado': 'alocada'}


## 2. Final boss / missão de equipe

A missão final exige uma dupla com as habilidades certas. Se a missão anterior não foi concluída, a equipe final fica pendente.

In [3]:
def concluir_missao_final(dados):
    plano = dados['plano'].copy()
    m1_ok = len(plano[0]['equipe']) > 0

    livres = []
    for nome in dados['disponiveis_para_final']:
        if nome in dados['ativos']:
            livres.append(nome)

    dupla = []
    estado = 'dependencia_pendente'

    if m1_ok:
        dupla = formar_coalizao(livres, {'força', 'cura'})
        estado = 'sem_capacidade'
        if dupla:
            estado = 'alocada'

    plano.append({'missao': 'M4', 'equipe': dupla, 'estado': estado, 'depende': ['M1']})
    resposta = dados.copy()
    resposta['plano'] = plano
    return resposta

resultado_final = concluir_missao_final(resultado)
for item in resultado_final['plano']:
    print(item)


{'missao': 'M1', 'equipe': ['Aria'], 'estado': 'alocada'}
{'missao': 'M2', 'equipe': ['Bruno'], 'estado': 'alocada'}
{'missao': 'M3', 'equipe': ['Cleo'], 'estado': 'alocada'}
{'missao': 'M4', 'equipe': ['Aria', 'Bruno'], 'estado': 'alocada', 'depende': ['M1']}
